# 03 · Retrieval-Augmented Generation (RAG)  *(CPU ok)*

**Approach 3 of 3.** Here we change **nothing** in the model. Instead, at question time we **look up
the most relevant documents** from our corpus and hand them to the model as context.

> **What RAG is good at:** new, changing, or **private facts** — exactly the Redlake University
> policies that no pretrained model could know. The model's weights never change; we just give it the
> right reading material at the right moment.

In [ ]:
import json, numpy as np, torch
from pathlib import Path
from sentence_transformers import SentenceTransformer
import faiss

ARTIFACTS = Path("artifacts")
corpus = json.load(open(ARTIFACTS / "corpus.json"))
docs = [f"{e['title']}: {e['text']}" for e in corpus]
print(len(docs), "documents to index")

### Build the search index (embeddings + FAISS)
We turn each document into a vector ("embedding") that captures its meaning, then store them in a
FAISS index so we can find the closest documents to any question.

In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
emb = embedder.encode(docs, normalize_embeddings=True).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)

faiss.write_index(index, str(ARTIFACTS / "rag.faiss"))
np.save(ARTIFACTS / "rag_docs.npy", np.array(docs, dtype=object))
print("Saved RAG index to", ARTIFACTS / "rag.faiss")

### Retrieve + answer
Given a question, find the top matching documents, put them in the prompt, and ask the instruct model
to answer **using only that context.**

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE_MODEL = "Qwen/Qwen3-1.7B"
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype="auto").to(device)

def retrieve(question, k=3):
    q = embedder.encode([question], normalize_embeddings=True).astype("float32")
    scores, idx = index.search(q, k)
    return [docs[i] for i in idx[0]]

def rag_answer(question, k=3, max_new_tokens=200):
    context = "\n".join(f"- {d}" for d in retrieve(question, k))
    msgs = [{"role": "user", "content":
             f"Use ONLY the context to answer. If it's not in the context, say so.\n\n"
             f"Context:\n{context}\n\nQuestion: {question} /no_think"}]
    prompt = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                     enable_thinking=False)
    ids = tok(prompt, return_tensors="pt").to(device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# A private fact only the corpus knows — RAG nails it; a plain model could not.
print(rag_answer("What is Redlake University's password policy?"))

**Takeaway:** no training at all — RAG answered a **private fact** correctly just by retrieving it.
That's its superpower: new and changing knowledge, with the model untouched. The trade-off is that it
only knows what's in the corpus, and answers depend on retrieving the right documents.